In [1]:
import pyodbc
from sqlalchemy import create_engine
import urllib
import warnings
from sqlalchemy.exc import SAWarning
import pandas as pd

In [ ]:
warnings.filterwarnings("ignore", category=SAWarning)

params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=edtech;"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}"
)

In [3]:
# Test connection
with engine.connect() as conn:
    print("Connection Successful")

Connection Successful


In [ ]:
df = pd.read_sql("select * from students", engine)
print(df.head())

   student_id student_fname student_lname           student_email  \
0           1          Amit        Sharma         amit1@gmail.com   
1           2          Amit        Sharma  amit.sharma1@gmail.com   
2           3         Priya          Nair   priya.nair2@gmail.com   
3           4         Rahul         Verma  rahul.verma3@gmail.com   
4           5         Sneha          Iyer   sneha.iyer4@gmail.com   

  student_phone student_alternate_phone  course_selected  \
0    9876543210                    None                1   
1    9876500001              9876510001                1   
2    9876500002              9876510002                2   
3    9876500003              9876510003                3   
4    9876500004              9876510004                4   

           enrolment_date  years_of_experience  batch_date  location_id  \
0 2026-02-18 11:55:08.440                    2  2025-03-01            1   
1 2026-02-18 11:56:13.860                    2  2025-03-01            1   

pandas.core.frame.DataFrame

In [5]:
from sqlalchemy.orm import sessionmaker

session = sessionmaker(engine)

session = session()

In [6]:
from sqlalchemy import text
result = session.execute(text("SELECT * FROM students"))

for row in result:
    print(row)

(1, 'Amit', 'Sharma', 'amit1@gmail.com', '9876543210', None, 1, datetime.datetime(2026, 2, 18, 11, 55, 8, 440000), 2, datetime.date(2025, 3, 1), 1, 1, 1)
(2, 'Amit', 'Sharma', 'amit.sharma1@gmail.com', '9876500001', '9876510001', 1, datetime.datetime(2026, 2, 18, 11, 56, 13, 860000), 2, datetime.date(2025, 3, 1), 1, 1, 1)
(3, 'Priya', 'Nair', 'priya.nair2@gmail.com', '9876500002', '9876510002', 2, datetime.datetime(2026, 2, 18, 11, 56, 13, 860000), 3, datetime.date(2025, 3, 1), 2, 2, 2)
(4, 'Rahul', 'Verma', 'rahul.verma3@gmail.com', '9876500003', '9876510003', 3, datetime.datetime(2026, 2, 18, 11, 56, 13, 860000), 1, datetime.date(2025, 3, 15), 4, 3, 3)
(5, 'Sneha', 'Iyer', 'sneha.iyer4@gmail.com', '9876500004', '9876510004', 4, datetime.datetime(2026, 2, 18, 11, 56, 13, 860000), 0, datetime.date(2025, 4, 1), 5, 10, 4)
(6, 'Arjun', 'Reddy', 'arjun.reddy5@gmail.com', '9876500005', '9876510005', 5, datetime.datetime(2026, 2, 18, 11, 56, 13, 860000), 4, datetime.date(2025, 4, 1), 6, 4, 1

In [7]:
result = session.execute(text("select count(student_id) as student_count from students"))

print(result.scalar())


11


In [ ]:
from sqlalchemy import select, func, table, column

stmt = select(func.count()).select_from(table("courses"))

with Session(engine) as session:
    result = session.execute(stmt)
    print(result.scalar())
    

8


In [ ]:
from sqlalchemy import (
    Column, Integer, String, Date, DateTime, 
    ForeignKey, CheckConstraint, UniqueConstraint,
    func, Index
)
from sqlalchemy.orm import relationship, declarative_base
from sqlalchemy.sql import expression
from sqlalchemy.types import Enum
from sqlalchemy import create_engine

Base = declarative_base()

In [ ]:
class Store(Base):
    __tablename__ = "stores"

    store_id = Column(Integer, primary_key=True)
    store_name = Column(String(100), nullable=False)
    city = Column(String(100))
    state = Column(String(100))
    created_at = Column(DateTime, server_default=func.now())

    orders = relationship("Order", back_populates="store")
    purchases = relationship("InventoryBought", back_populates="store")

In [ ]:
class Inventory(Base):
    __tablename__ = "inventory"

    inventory_id = Column(Integer, primary_key=True)
    store_id = Column(Integer, ForeignKey("stores.store_id"))
    category = Column(String(100), nullable=False)
    opening_stock = Column(Integer, nullable=False)
    created_at = Column(Date, nullable=False)

    store = relationship("Store")

In [ ]:
class Order(Base):
    __tablename__ = "orders"

    order_id = Column(Integer, primary_key=True)
    store_id = Column(Integer, ForeignKey("stores.store_id"))
    category = Column(String(100), nullable=False)
    qty = Column(Integer, nullable=False)
    order_date = Column(Date, nullable=False)

    __table_args__ = (
        CheckConstraint("qty > 0", name="check_qty_positive"),
        Index("idx_orders_date", "store_id", "category", "order_date"),
    )

    store = relationship("Store", back_populates="orders")

In [ ]:
class InventoryBought(Base):
    __tablename__ = "inventory_bought"

    purchase_id = Column(Integer, primary_key=True)
    store_id = Column(Integer, ForeignKey("stores.store_id"))
    category = Column(String(100), nullable=False)
    bought_qty = Column(Integer, nullable=False)
    bought_date = Column(Date, nullable=False)

    __table_args__ = (
        CheckConstraint("bought_qty > 0", name="check_bought_positive"),
        Index("idx_bought_date", "store_id", "category", "bought_date"),
    )

    store = relationship("Store", back_populates="purchases")

In [ ]:
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
Base.metadata.create_all(engine)